In [1]:
from spark_start import spark
import os, time
import pandas as pd

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(PROJECT_ROOT)

df = spark.read.parquet(f"{PROJECT_ROOT}/data/processed/final")
print(df.count())


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/16 01:02:26 WARN Utils: Your hostname, Saideep-Reddys-Mac.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.2 instead (on interface en0)
26/02/16 01:02:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/16 01:02:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/16 01:02:28 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/02/16 01:02:28 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/02/16 01:02:28 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/02/16 01:02:28 WAR

Spark started successfully


2540047


In [2]:
small = df.limit(100000)
medium = df.limit(500000)
large = df

small.cache(); small.count()
medium.cache(); medium.count()
large.cache(); large.count()


2540047

In [3]:
from pyspark.ml.classification import RandomForestClassifier

def train_time(data):

    rf = RandomForestClassifier(
        labelCol="label_binary",
        featuresCol="features",
        numTrees=20,
        maxDepth=6
    )

    start = time.time()
    model = rf.fit(data)
    end = time.time()

    return end - start


In [4]:
results = []

for name, dataset in [("100K", small), ("500K", medium), ("FULL", large)]:
    t = train_time(dataset)
    print(name, "time:", t)
    results.append((name, t))


100K time: 3.231980085372925


500K time: 13.068938970565796


FULL time: 29.847646951675415


In [5]:
scale_df = pd.DataFrame(results, columns=["Dataset_Size","Training_Time_sec"])
scale_df.to_csv("tableau/csv_exports/scalability.csv", index=False)
scale_df


,Dataset_Size,Training_Time_sec
0,100K,3.231980
1,500K,13.068939
2,FULL,29.847647
